In [7]:
# -*- coding: utf-8 -*-
"""
EDA summary for person 18 (1-indexed) using Big-Five item responses.

What it prints (three sentences):
1) Specific strengths (domains with higher percentiles or, if none, relative strengths) + top three distinctive high items.
2) Aspects that are average and/or weak + three comparatively low items.
3) A single concluding statement summarizing the overall profile.

Files required in the working directory:
- data-sample.csv  (the dataset)
- map.json         (maps item codes like 'EXT1' -> human-readable phrase)

Assumptions:
- Big-Five items are the first 50 columns named EXT1..EXT10, EST1..EST10, AGR1..AGR10, CSN1..CSN10, OPN1..OPN10.
- Domain scores = mean of the 10 items in each domain.
- Percentiles are computed relative to the provided sample.
"""

import pandas as pd
import json
import numpy as np

# -------------------------------------------------------------------
# 1) Load data
# -------------------------------------------------------------------
CSV_PATH = '/data/demo_data/big_five/data-sample.csv'
MAP_PATH = '/data/demo_data/big_five/map.json'

df = pd.read_csv(CSV_PATH)
with open(MAP_PATH, 'r') as f:
    item_map = json.load(f)

# -------------------------------------------------------------------
# 2) Identify item columns and domain groupings
# -------------------------------------------------------------------
prefixes = ['EXT', 'EST', 'AGR', 'CSN', 'OPN']
item_cols = []
for p in prefixes:
    for i in range(1, 11):
        col = f"{p}{i}"
        if col in df.columns:
            item_cols.append(col)
# keep exactly 50 items (in case extra similarly named columns exist)
item_cols = item_cols[:50]

domain_groups = {
    'EXT': {'name': 'Extraversion',          'cols': [c for c in item_cols if c.startswith('EXT')]},
    'EST': {'name': 'Emotional Stability',   'cols': [c for c in item_cols if c.startswith('EST')]},
    'AGR': {'name': 'Agreeableness',         'cols': [c for c in item_cols if c.startswith('AGR')]},
    'CSN': {'name': 'Conscientiousness',     'cols': [c for c in item_cols if c.startswith('CSN')]},
    'OPN': {'name': 'Openness',              'cols': [c for c in item_cols if c.startswith('OPN')]}
}

# -------------------------------------------------------------------
# 3) Compute domain means and sample-relative percentiles
# -------------------------------------------------------------------
for k, v in domain_groups.items():
    df[f'{k}__mean'] = df[v['cols']].mean(axis=1)

person_idx = 17  # "person 18" (1-indexed) -> iloc[17]

domain_stats = {}
for k, v in domain_groups.items():
    vals = df[f'{k}__mean']
    pct = vals.rank(pct=True) * 100.0  # sample-relative percentile
    domain_stats[k] = {
        'name': v['name'],
        'pct': float(pct.iloc[person_idx]),
        'mean': float(vals.iloc[person_idx])
    }

# -------------------------------------------------------------------
# 4) Surface distinctive items for the target person via z-scores
# -------------------------------------------------------------------
item_z = (df[item_cols] - df[item_cols].mean()) / df[item_cols].std(ddof=0)
person_item_z = item_z.iloc[person_idx].replace([np.inf, -np.inf], np.nan).fillna(0)

top_items = person_item_z.sort_values(ascending=False).head(3).index.tolist()
bot_items = person_item_z.sort_values(ascending=True).head(3).index.tolist()

def phrase(code: str) -> str:
    """Map item code to a readable phrase when available."""
    return item_map.get(code, code)

# -------------------------------------------------------------------
# 5) Classify domains into strengths / average / weaknesses
# -------------------------------------------------------------------
strong_thr, weak_thr = 65, 35
avg_low, avg_high = 40, 60

sorted_k = sorted(domain_stats.keys(), key=lambda k: domain_stats[k]['pct'], reverse=True)
strengths   = [(domain_stats[k]['name'], domain_stats[k]['pct']) for k in sorted_k if domain_stats[k]['pct'] >= strong_thr]
weaknesses  = [(domain_stats[k]['name'], domain_stats[k]['pct']) for k in sorted_k if domain_stats[k]['pct'] <= weak_thr]
averages    = [(domain_stats[k]['name'], domain_stats[k]['pct']) for k in sorted_k if avg_low <= domain_stats[k]['pct'] <= avg_high]

# -------------------------------------------------------------------
# 6) Compose the three required sentences
# -------------------------------------------------------------------
# Strengths sentence
if strengths:
    top_str = ', '.join([f"{n} ({int(p)}th pct)" for n, p in strengths[:3]])
    strengths_sentence = (
        f"Person 18's clearest strengths are in {top_str}; "
        f"their most pronounced item tendencies are {phrase(top_items[0])}, {phrase(top_items[1])}, and {phrase(top_items[2])}."
    )
else:
    # No domains above the strong threshold – describe the relative top domains.
    rel = [(domain_stats[k]['name'], int(domain_stats[k]['pct'])) for k in sorted_k[:2]]
    strengths_sentence = (
        f"Person 18's relative strengths (within this sample) are {rel[0][0]} ({rel[0][1]}th pct) and {rel[1][0]} ({rel[1][1]}th pct); "
        f"their most pronounced item tendencies are {phrase(top_items[0])}, {phrase(top_items[1])}, and {phrase(top_items[2])}."
    )

# Average/weakness sentence
segments = []
if averages:
    if len(averages) == 1:
        segments.append(f"average in {averages[0][0]} ({int(averages[0][1])}th pct)")
    else:
        mid = ', '.join([f"{n} ({int(p)}th pct)" for n, p in averages[:-1]]) + f" and {averages[-1][0]} ({int(averages[-1][1])}th pct)"
        segments.append(f"average in {mid}")
if weaknesses:
    if len(weaknesses) == 1:
        segments.append(f"relatively lower on {weaknesses[0][0]} ({int(weaknesses[0][1])}th pct)")
    else:
        low = ', '.join([f"{n} ({int(p)}th pct)" for n, p in weaknesses[:-1]]) + f" and {weaknesses[-1][0]} ({int(weaknesses[-1][1])}th pct)"
        segments.append(f"relatively lower on {low}")
if not segments:
    segments.append("mostly within typical ranges across domains")

weak_sentence = (
    f"They are {'; '.join(segments)}; comparatively lower-scoring items include "
    f"{phrase(bot_items[0])}, {phrase(bot_items[1])}, and {phrase(bot_items[2])}."
)

# Concluding single-sentence summary
high_list = [n for n, _ in strengths] or [domain_stats[sorted_k[0]]['name']]  # ensure at least the top domain
avg_list = [n for n, _ in averages]
low_list = [n for n, _ in weaknesses]

summary_sentence = (
    f"Overall, person 18 trends higher in {', '.join(high_list)}, "
    f"sits around average in {', '.join(avg_list) if avg_list else 'few areas'}, "
    f"and lower in {', '.join(low_list) if low_list else 'very few areas'}."
)

# -------------------------------------------------------------------
# 7) Output the three sentences
# -------------------------------------------------------------------
print(strengths_sentence)
print(weak_sentence)
print(summary_sentence)

Person 18's relative strengths (within this sample) are Agreeableness (55th pct) and Extraversion (39th pct); their most pronounced item tendencies are they are relaxed most of the time, they feel little concern for others, and they like order.
They are average in Agreeableness (55th pct); relatively lower on Openness (29th pct), Emotional Stability (12th pct) and Conscientiousness (5th pct); comparatively lower-scoring items include they make people feel at ease, they worry about things, and they are quick to understand things.
Overall, person 18 trends higher in Agreeableness, sits around average in Agreeableness, and lower in Openness, Emotional Stability, Conscientiousness.
